# Feature Engineering cho Dầu, Cửa hàng & Giao dịch (Oil, Store & Transaction Features)

Notebook này triển khai ba nhóm feature từ file `feature_oil_and_store.py`:

## Các Feature được tạo ra

### Oil Features (Đặc trưng Giá dầu)
| Feature | Mô tả |
|---|---|
| `oil_price` | Giá dầu WTI đã ffill missing values |
| `oil_price_lag_7` | Giá dầu 7 ngày trước |
| `oil_price_lag_14` | Giá dầu 14 ngày trước |
| `oil_price_rolling_mean_28` | Rolling mean 28 ngày (shift-1 safe) |
| `oil_price_change_pct` | Phần trăm thay đổi giá dầu so với tuần trước |

### Store Features (Đặc trưng Cửa hàng)
| Feature | Mô tả |
|---|---|
| `store_type_enc` | Label encode loại cửa hàng (fit on train only) |
| `city_freq` | Frequency encode thành phố |
| `state_freq` | Frequency encode bang/tỉnh |

### Transaction Features (Chưa triển khai trong file gốc)
| Feature | Mô tả |
|---|---|
| `transactions_lag_7` | Số giao dịch cùng ngày tuần trước |
| `transactions_rolling_mean_7` | Rolling mean giao dịch 7 ngày |

## Tổng quan Pipeline
```
Load Data → Oil Features (ffill + lag + rolling)
    → Store Encoding (label encode + frequency encode)
    → Transaction Features (TODO) → Save Output
```

## 1. Import Thư viện

Sử dụng **pandas** cho thao tác dữ liệu và **LabelEncoder** từ scikit-learn để mã hóa `store_type`.

In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

## 2. Đọc Dữ liệu

Pipeline cần các bộ dữ liệu sau:
- **`train_cleaned.csv`** — dữ liệu train chính (cần `date`, `store_nbr`).
- **`test_cleaned.csv`** — dữ liệu test (cần cho store encoding fit on train only).
- **`oil.csv`** — giá dầu WTI hàng ngày.
- **`stores_cleaned.csv`** — thông tin cửa hàng (`store_type`, `city`, `state`).

In [11]:
BASE_PATH = r'D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data'

df_train  = pd.read_csv(rf'{BASE_PATH}\processed\train_cleaned.csv')
df_test   = pd.read_csv(rf'{BASE_PATH}\processed\test_cleaned.csv')
df_oil    = pd.read_csv(rf'{BASE_PATH}\raw\oil.csv')
df_stores = pd.read_csv(rf'{BASE_PATH}\processed\stores_cleaned.csv')

print(f'Train shape  : {df_train.shape}')
print(f'Test shape   : {df_test.shape}')
print(f'Oil shape    : {df_oil.shape}')
print(f'Stores shape : {df_stores.shape}')
df_oil.head(3)

Train shape  : (3000888, 6)
Test shape   : (28512, 5)
Oil shape    : (1218, 2)
Stores shape : (54, 6)


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97


---
## Phần 1 — Oil Features

Giá dầu WTI (`dcoilwtico`) ảnh hưởng trực tiếp đến nền kinh tế Ecuador (quốc gia xuất khẩu dầu). Ta tạo các feature lag và rolling từ giá dầu, sử dụng `ffill` để xử lý missing values (ngày không giao dịch).

Hàm `create_oil_features` đã được **sửa lỗi return** so với file gốc — `return oil` giờ nằm đúng vị trí ở cuối hàm.

In [12]:
def create_oil_features(oil_df):
    oil = oil_df.copy().rename(columns={"dcoilwtico":"oil_price"})
    oil["date"] = pd.to_datetime(oil["date"])
    oil = oil[["date","oil_price"]].sort_values("date").reset_index(drop=True)

    full_date_range = pd.DataFrame({"date": pd.date_range(start=oil["date"].min(), end=oil["date"].max(), freq="D")})
    oil = full_date_range.merge(oil, on="date", how="left")

    oil["oil_price"] = oil["oil_price"].ffill()
    oil["oil_price"] = oil["oil_price"].bfill()

    oil["oil_price_lag_7"]  = oil["oil_price"].shift(7)
    oil["oil_price_lag_14"] = oil["oil_price"].shift(14)
    oil["oil_price_rolling_mean_28"] = oil["oil_price"].shift(1).rolling(28, min_periods=7).mean()
    oil["oil_price_change_pct"] = oil["oil_price"].pct_change(periods=7)

    return oil

# Test the function
oil_featured = create_oil_features(df_oil)
print(f'Oil featured shape: {oil_featured.shape}')
print(f'Columns: {oil_featured.columns.tolist()}')
oil_featured.head(10)

Oil featured shape: (1704, 6)
Columns: ['date', 'oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct']


,date,oil_price,oil_price_lag_7,oil_price_lag_14,oil_price_rolling_mean_28,oil_price_change_pct
0,2013-01-01,93.14,NaN,NaN,NaN,NaN
1,2013-01-02,93.14,NaN,NaN,NaN,NaN
2,2013-01-03,92.97,NaN,NaN,NaN,NaN
3,2013-01-04,93.12,NaN,NaN,NaN,NaN
4,2013-01-05,93.12,NaN,NaN,NaN,NaN
5,2013-01-06,93.12,NaN,NaN,NaN,NaN
6,2013-01-07,93.20,NaN,NaN,NaN,NaN
7,2013-01-08,93.21,93.14,NaN,93.115714,0.000752
8,2013-01-09,93.08,93.14,NaN,93.127500,-0.000644
9,2013-01-10,93.81,92.97,NaN,93.122222,0.009035


---
## Phần 2 — Store Encoding

Mã hóa các đặc trưng cửa hàng:
- **Label encode `store_type`** — fit trên train, transform trên cả train và test để tránh data leakage.
- **Frequency encode `city` và `state`** — tính tần suất xuất hiện trên train, map sang cả hai tập.

Hàm `encode_store_features` đã được **tách ra khỏi** `create_oil_features` (sửa lỗi nested function trong file gốc).

In [13]:
def encode_store_features(train, test):

    train, test = train.copy(), test.copy()

    # 1. Label encode store_type (fit on train only)
    le = LabelEncoder()
    train["store_type_enc"] = le.fit_transform(train["type"])
    test["store_type_enc"] = le.transform(test["type"])

    # 2. Frequency encode city/state
    for col in ["city", "state"]:
        freq = train[col].value_counts(normalize=True)
        train[f"{col}_freq"] = train[col].map(freq)
        test[f"{col}_freq"] = test[col].map(freq).fillna(0)

    # 3. cluster keep the same

    return train, test


# Merge store info into train/test before encoding
store_cols = ['store_nbr', 'type', 'city', 'state']
train_with_store = df_train.merge(df_stores[store_cols], on='store_nbr', how='left')
test_with_store  = df_test.merge(df_stores[store_cols], on='store_nbr', how='left')

train_encoded, test_encoded = encode_store_features(train_with_store, test_with_store)

print(f'Train encoded shape: {train_encoded.shape}')
print(f'Test encoded shape : {test_encoded.shape}')
print(f'\nNew columns: store_type_enc, city_freq, state_freq')
train_encoded[['store_nbr', 'type', 'store_type_enc', 'city_freq', 'state_freq']].drop_duplicates('store_nbr').head(10)

Train encoded shape: (3000888, 12)
Test encoded shape : (28512, 11)

New columns: store_type_enc, city_freq, state_freq


,store_nbr,type,store_type_enc,city_freq,state_freq
0,1,D,3,0.333333,0.351852
33,10,C,2,0.333333,0.351852
66,11,B,1,0.018519,0.351852
99,12,C,2,0.037037,0.037037
132,13,C,2,0.037037,0.037037
165,14,C,2,0.018519,0.018519
198,15,C,2,0.018519,0.018519
231,16,C,2,0.055556,0.055556
264,17,C,2,0.333333,0.351852
297,18,B,1,0.333333,0.351852


---
## Phần 3 — Transaction Features 

Theo mô tả yêu cầu, số lượng giao dịch (transactions) mỗi ngày tại mỗi cửa hàng là proxy tốt cho lượng khách hàng. Cần tạo hai feature:

| Feature | Mô tả |
|---|---|
| `transactions_lag_7` | Số giao dịch cùng ngày tuần trước (same day last week) |
| `transactions_rolling_mean_7` | Rolling average số giao dịch 7 ngày gần nhất |

**Lưu ý**: Chỉ được dùng **lag transactions** để tránh data leakage — không dùng giá trị transactions của ngày hiện tại.



## Phần 4 — Danh sách tên Feature

Định nghĩa các hằng số tên feature để dùng trong các bước downstream (chọn feature, xác thực đầu ra).

In [14]:
OIL_FEATURE_NAMES = [
    'oil_price',
    'oil_price_lag_7',
    'oil_price_lag_14',
    'oil_price_rolling_mean_28',
    'oil_price_change_pct',
]

STORE_FEATURE_NAMES = [
    'store_type_enc',
    'city_freq',
    'state_freq',
]

# TODO: Uncomment khi transaction features được triển khai
# TRANSACTION_FEATURE_NAMES = [
#     'transactions_lag_7',
#     'transactions_rolling_mean_7',
# ]

ALL_OIL_STORE_FEATURES = OIL_FEATURE_NAMES + STORE_FEATURE_NAMES

print(f'Oil features   : {len(OIL_FEATURE_NAMES)}')
print(f'Store features : {len(STORE_FEATURE_NAMES)}')
print(f'Total features : {len(ALL_OIL_STORE_FEATURES)}')
print(f'\nAll features: {ALL_OIL_STORE_FEATURES}')

Oil features   : 5
Store features : 3
Total features : 8

All features: ['oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct', 'store_type_enc', 'city_freq', 'state_freq']


## Phần 5 — Chạy Pipeline

Áp dụng toàn bộ pipeline:
1. Tạo oil features và merge vào train.
2. Encode store features cho cả train và test.
3. Hiển thị kết quả tổng hợp.

In [15]:
# --- Step 1: Oil features ---
oil_featured = create_oil_features(df_oil)

# Merge oil features vào train
df_train['date'] = pd.to_datetime(df_train['date'])
df_result = df_train.merge(oil_featured[['date'] + OIL_FEATURE_NAMES], on='date', how='left')

print(f'After oil merge shape: {df_result.shape}')
print(f'Oil NaN counts:')
print(df_result[OIL_FEATURE_NAMES].isna().sum())

# --- Step 2: Store encoding ---
store_cols = ['store_nbr', 'type', 'city', 'state']
df_result = df_result.merge(df_stores[store_cols], on='store_nbr', how='left')

df_test_tmp = df_test.copy()
df_test_tmp = df_test_tmp.merge(df_stores[store_cols], on='store_nbr', how='left')

df_result, df_test_out = encode_store_features(df_result, df_test_tmp)

print(f'\nAfter store encoding shape: {df_result.shape}')
print(f'\n=== Feature Summary ===')
print(df_result[ALL_OIL_STORE_FEATURES].describe().T[['mean', 'min', 'max']])
print(f'\n=== Sample rows ===')
df_result[['date', 'store_nbr'] + ALL_OIL_STORE_FEATURES].head(10)

After oil merge shape: (3000888, 11)
Oil NaN counts:
oil_price                        0
oil_price_lag_7              12474
oil_price_lag_14             24948
oil_price_rolling_mean_28    12474
oil_price_change_pct         12474
dtype: int64

After store encoding shape: (3000888, 17)

=== Feature Summary ===
                                mean        min         max
oil_price                  67.924899  26.190000  110.620000
oil_price_lag_7            68.009296  26.190000  110.620000
oil_price_lag_14           68.083719  26.190000  110.620000
oil_price_rolling_mean_28  68.202760  30.210000  108.076429
oil_price_change_pct       -0.001698  -0.171989    0.287284
store_type_enc              2.000000   0.000000    4.000000
city_freq                   0.149520   0.018519    0.333333
state_freq                  0.182442   0.018519    0.351852

=== Sample rows ===


,date,store_nbr,oil_price,oil_price_lag_7,oil_price_lag_14,oil_price_rolling_mean_28,oil_price_change_pct,store_type_enc,city_freq,state_freq
0,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
1,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
2,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
3,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
4,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
5,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
6,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
7,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
8,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
9,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852


## Lưu Kết quả

Lưu dataframe đã được bổ sung oil và store features vào thư mục processed.

In [16]:
OUTPUT_PATH = rf'{BASE_PATH}\processed\train_oil_store_features.csv'
df_result.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to: {OUTPUT_PATH}')
print(f'Shape   : {df_result.shape}')

KeyboardInterrupt: 